# Visualize — the old figures, on three cohorts

Reads what `05_decompose.ipynb` wrote. Six figures from
`2. decomposition vizualization.ipynb`, each written once as a function so the
cohort / embedding / colour is an argument instead of a copy-paste, plus one
new figure: the latent space coloured by symptom score.

`ds002837` and `cneuromod` are the cohorts the model was fit on; `camcan` was
projected. Differences between camcan and the other two partly reflect
acquisition (TR 2.47 against 1.0 and 1.49), so read *within*-cohort structure,
not the offset between them.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.io as pio
    # JupyterLab renders plotly's own mimetype only with the labextension
    # installed; without it fig.show() produces a silently empty output. The
    # iframe renderer needs no extension, and keeps plotly.js out of the
    # .ipynb -- embedding it is how a previous notebook reached 50 MB.
    pio.renderers.default = "iframe_connected"
    HAVE_PLOTLY = True
except ImportError:                      # the image may not ship plotly
    px, HAVE_PLOTLY = None, False
    print("plotly absent -- 3D scatters fall back to matplotlib (static)")

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))
ATLAS = "harvardoxford"
WINDOW_S = 30
CLUSTER = "ThresholdCluster_pca3_512"
PHENO = Path.cwd().parent / "config" / "phenotype" / "camcan_phenotype.csv"

def load_latents(window_s="*"):
    # Latents for one window size, or every one written so far.
    paths = sorted((ROOT / "latents" / f"atlas={ATLAS}").glob(
        f"window_s={window_s}/cohort=*/data.parquet"))
    if not paths:
        raise FileNotFoundError(
            f"no latents under latents/atlas={ATLAS}. Run the decomposition "
            f"first:  sbatch --array=0-0 slurm/04_decompose.sbatch {ATLAS} {WINDOW_S}")
    return pd.concat([pd.read_parquet(q).assign(
        **dict(s.split("=", 1) for s in q.parts if "=" in s)) for q in paths],
        ignore_index=True)

allw = load_latents()                       # every window size on disk
allw["window_s"] = allw["window_s"].astype(float)
lat = allw[allw["window_s"] == WINDOW_S]    # the one most cells below use

# `role` says whether the model was FIT on this cohort or only applied to it;
# `model_hash` says which fit produced the row. Two rows with different hashes
# are not comparable, whatever the paths look like.
print(allw.groupby(["window_s", "cohort", "role", "model_hash"])
          .size().rename("windows").to_string())
print(f"\nWINDOW_S={WINDOW_S}: {len(lat):,} windows, "
      f"{CLUSTER} has {lat[CLUSTER].nunique()} occupied cells")
lat.head(3)

## The plotting functions

Two of them. Every 3D figure below is one `scatter3` call; the old notebook had
sixteen near-identical cells.

In [ ]:
def scatter3(df, embed="pca", n=3, color="sub", title=None, size=2, **kw):
    """3D scatter of any embedding, coloured by any column or boolean mask."""
    cols = [f"{embed}{j}/{n}" for j in range(3)]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"skipped: {missing} not in the latents ({embed}{n} not fit?)")
        return
    label = title or (f"{embed}{n} | colour = "
                      f"{color if isinstance(color, str) else 'mask'}")
    if HAVE_PLOTLY:
        fig = px.scatter_3d(df, x=cols[0], y=cols[1], z=cols[2], color=color,
                            width=800, height=600, title=label, **kw)
        fig.update_traces(marker=dict(size=size))
        fig.show()
        return
    # Static fallback: same three axes, colour resolved the same way.
    c = df[color] if isinstance(color, str) else color
    if c.dtype == object or str(c.dtype) in ("bool", "category", "string"):
        codes, uniq = pd.factorize(c)
    else:
        codes = pd.to_numeric(c, errors="coerce")
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(projection="3d")
    sc = ax.scatter(df[cols[0]], df[cols[1]], df[cols[2]], c=codes,
                    s=size * 2, cmap="viridis", alpha=0.6)
    ax.set_xlabel(cols[0], fontsize=7); ax.set_ylabel(cols[1], fontsize=7)
    ax.set_zlabel(cols[2], fontsize=7); ax.set_title(label, fontsize=9)
    fig.colorbar(sc, ax=ax, shrink=0.6)
    fig.tight_layout(); plt.show()


def timecourse(df, cols, group=("task", "window_id"), x="stimulus_start_s"):
    """Per-window scatter + across-subject mean +/- SD, one row per task."""
    tasks = sorted(df["task"].unique())
    fig, axes = plt.subplots(len(tasks), len(cols),
                             figsize=(4.2 * len(cols), 2.6 * len(tasks)),
                             squeeze=False)
    for i, task in enumerate(tasks):
        d = df[df["task"] == task]
        agg = d.groupby(list(group))[list(cols) + [x]].agg(["mean", "std"])
        for j, col in enumerate(cols):
            ax = axes[i][j]
            ax.scatter(d[x], d[col], alpha=0.05, color="black", s=6)
            xs, m, s = agg[(x, "mean")], agg[(col, "mean")], agg[(col, "std")]
            ax.plot(xs, m, linewidth=1.5)
            ax.fill_between(xs, m - s, m + s, alpha=0.15)
            ax.set_title(f"{task} | {col}  sd={s.mean():.2f}", fontsize=8)
            ax.grid(True, linestyle="--", alpha=0.2)
    fig.tight_layout()

## 1–2. PCA space, by subject and by cluster

In [ ]:
for cohort in sorted(lat["cohort"].unique()):
    d = lat[lat["cohort"] == cohort]
    # sub is a string, so plotly needs a qualitative palette with enough
    # entries; only reachable when plotly is actually installed.
    kw = dict(color_discrete_sequence=px.colors.qualitative.Alphabet) if HAVE_PLOTLY else {}
    scatter3(d, "pca", 3, color="sub", title=f"{cohort} | pca3 | subject", **kw)

In [ ]:
for cohort in sorted(lat["cohort"].unique()):
    scatter3(lat[lat["cohort"] == cohort], "pca", 3, color=CLUSTER,
             title=f"{cohort} | pca3 | {CLUSTER}")

## 3. One subject against the rest

In [ ]:
cohort = "ds002837"
d = lat[lat["cohort"] == cohort]
one = sorted(d["sub"].unique())[0]
scatter3(d, "pca", 3, color=(d["sub"] == one).rename("is_target"),
         title=f"{cohort} | pca3 | sub-{one} vs other subjects")

## 4. One task against the rest

In [ ]:
d = lat[lat["cohort"] == "ds002837"]
for task in sorted(d["task"].unique()):
    scatter3(d, "pca", 3, color=(d["task"] == task).rename("is_task"),
             title=f"ds002837 | pca3 | {task} vs other tasks")

## 5. UMAP — every window size, every cohort

`PLOT_SAMPLE` caps each panel: plotly's 3D renderer struggles well before
100,000 points, and camcan alone is ~48,000 windows at 30 s.

UMAP coordinates are **not comparable across window sizes** — each is its own
fit, and UMAP has no canonical orientation or scale. Read structure within a
panel, not the difference between panels.

In [ ]:
PLOT_SAMPLE = 20_000

def sub(d, n=PLOT_SAMPLE, seed=0):
    return d.sample(n, random_state=seed) if n and len(d) > n else d

for w in sorted(allw["window_s"].unique()):
    for cohort in sorted(allw["cohort"].unique()):
        d = sub(allw[(allw["window_s"] == w) & (allw["cohort"] == cohort)])
        if d.empty:
            continue
        scatter3(d, "umap", 3, color=CLUSTER,
                 title=f"{cohort} | {w:.0f}s | umap3 | {CLUSTER}")

In [ ]:
# All three cohorts in one plot: this is the "do the cohorts look alike"
# figure. camcan is `role=projected`, so a tight blob off to one side is the
# acquisition difference (TR 2.47 against 1.0 and 1.49) before it is biology.
for w in sorted(allw["window_s"].unique()):
    scatter3(sub(allw[allw["window_s"] == w], 30_000), "umap", 3,
             color="cohort", title=f"{w:.0f}s | umap3 | cohort")

## 6. One subject's latents over time

In [ ]:
cohort = "ds002837"
d = lat[lat["cohort"] == cohort]
one = sorted(d["sub"].unique())[0]
task = sorted(d.loc[d["sub"] == one, "task"].unique())[0]
sel = d[(d["sub"] == one) & (d["task"] == task)].sort_values("stimulus_start_s")

fig, ax = plt.subplots(figsize=(10, 4))
for col in [f"pca{j}/3" for j in range(3)]:
    ax.plot(sel["stimulus_start_s"], sel[col], label=col, linewidth=1)
ax.set_xlabel("stimulus time (s)"); ax.set_ylabel("latent")
ax.set_title(f"{cohort} sub-{one} | {task}", fontsize=10)
ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)

ax2 = ax.twinx()
ax2.step(sel["stimulus_start_s"], sel[CLUSTER], color="grey", alpha=0.4,
         where="post", linewidth=0.8)
ax2.set_ylabel(CLUSTER, color="grey")
fig.tight_layout()

## 7. All subjects, all tasks — inter-subject similarity

The one figure that aggregates. Black dots are individual windows; the line is
the across-subject mean at each window of the stimulus grid, the band is ±1 SD.
A narrow band means subjects are in the same latent place at the same moment of
the film.

In [ ]:
timecourse(lat[lat["cohort"] == "ds002837"],
           cols=[f"pca{j}/3" for j in range(3)] + [CLUSTER])

In [ ]:
# camcan: one task, 648 subjects -- the tightest test of inter-subject
# similarity in the set, and the cohort the model never saw.
timecourse(lat[lat["cohort"] == "camcan"],
           cols=[f"pca{j}/3" for j in range(3)] + [CLUSTER])

## 8. NEW — the latent space coloured by symptom

camcan only: HADS anxiety and depression, from the Stage 1 home interview,
joined per subject. Values above 900 are the archive's missing sentinel and are
dropped, not plotted as scores.

This is a picture, not a test. The symptom analysis — dispersion, transition
matrices, model selection — is deliberately not here.

In [ ]:
if not PHENO.exists():
    print(f"no phenotype at {PHENO}\n"
          f"  build it: python tools/make_phenotype.py --cohort camcan "
          f"--source approved_data.tsv --sub-column CCID "
          f"--sex-column homeint_sex "
          f"--keep additional_HADS_anxiety,additional_HADS_depression")
    pheno = None
else:
    pheno = pd.read_csv(PHENO, dtype={"sub": str})
    hads = [c for c in pheno.columns if "hads" in c.lower()]
    for c in hads:                       # 9999 is 'missing', not a score
        pheno[c] = pd.to_numeric(pheno[c], errors="coerce").where(lambda s: s <= 900)
    print(f"{len(pheno)} phenotype row(s), HADS columns: {hads}")
    print(pheno[hads].describe().round(2).to_string())

In [ ]:
if pheno is not None:
    cam = lat[lat["cohort"] == "camcan"].merge(pheno, on="sub", how="left")
    for col in [c for c in cam.columns if "hads" in c.lower()]:
        n = cam.loc[cam[col].notna(), "sub"].nunique()
        print(f"{col}: {n} subject(s) with a score")
        scatter3(cam[cam[col].notna()], "pca", 3, color=col,
                 title=f"camcan | pca3 | {col}  (n={n} subjects)",
                 color_continuous_scale="Viridis")

In [ ]:
# The same thing as a per-subject summary, which is easier to read than
# 48,000 windows: each subject's mean position, coloured by symptom.
if pheno is not None:
    col = "additional_HADS_anxiety"
    if col in cam.columns:
        per_sub = (cam.groupby("sub")
                      .agg({**{f"pca{j}/3": "mean" for j in range(3)}, col: "first"})
                      .dropna(subset=[col]).reset_index())
        scatter3(per_sub, "pca", 3, color=col, size=5,
                 title=f"camcan | subject-mean pca3 | {col}",
                 color_continuous_scale="Viridis")